In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import plotly.graph_objects as go
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pmdarima import auto_arima
import warnings
warnings.filterwarnings('ignore')

# Set styling for plots
sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')

In [ ]:
# Download Bitcoin data from Yahoo Finance
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365*2)).strftime('%Y-%m-%d')  # 2 years of data

# Download BTC-USD data
df = yf.download('BTC-USD', start=start_date, end=end_date)

# Reset index to make Date a column
df = df.reset_index()

# Display the first few rows of the data
print(f"Data range: {df['Date'].min()} to {df['Date'].max()}")
df.head()

In [ ]:
# Create interactive plot with Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['Close'], mode='lines', name='Bitcoin Close Price'))
fig.update_layout(
    title='Bitcoin Price History',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_dark'
)
fig.show()

# Recent trend - last 30 days
recent_df = df.tail(30).copy()
plt.figure(figsize=(14, 7))
plt.plot(recent_df['Date'], recent_df['Close'])
plt.title('Bitcoin Price - Last 30 Days')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Create a copy of the dataframe for feature engineering
data = df.copy()

# Add date-based features
data['day_of_week'] = data['Date'].dt.dayofweek
data['month'] = data['Date'].dt.month
data['year'] = data['Date'].dt.year
data['day'] = data['Date'].dt.day

# Create technical indicators
# Moving Averages
data['MA7'] = data['Close'].rolling(window=7).mean()
data['MA14'] = data['Close'].rolling(window=14).mean()
data['MA30'] = data['Close'].rolling(window=30).mean()

# Exponential Moving Averages
data['EMA12'] = data['Close'].ewm(span=12, adjust=False).mean()
data['EMA26'] = data['Close'].ewm(span=26, adjust=False).mean()

# MACD (Moving Average Convergence Divergence)
data['MACD'] = data['EMA12'] - data['EMA26']
data['MACD_signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

# Relative Strength Index (RSI) - Simplified calculation
delta = data['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
rs = gain / loss
data['RSI'] = 100 - (100 / (1 + rs))

# Volatility (Standard Deviation over 14 days)
data['Volatility'] = data['Close'].rolling(window=14).std()

# Price change percentage
data['Price_Change'] = data['Close'].pct_change() * 100

# Volume change percentage
data['Volume_Change'] = data['Volume'].pct_change() * 100

# Display the enhanced dataframe
data = data.dropna().reset_index(drop=True)
data.head()

In [ ]:
# Create a dataframe for modeling
model_data = data[['Date', 'Close', 'Volume', 'MA7', 'MA14', 'MA30', 'MACD', 'RSI', 'Volatility', 'Price_Change', 'Volume_Change']].copy()

# Set Date as index for time series models
model_data.set_index('Date', inplace=True)

# Split the data - keep the last 30 days for testing
test_size = 30
train_data = model_data[:-test_size]
test_data = model_data[-test_size:]

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")

# Plot training and testing data
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data['Close'], label='Training Data')
plt.plot(test_data.index, test_data['Close'], label='Testing Data')
plt.title('Training and Testing Data Split')
plt.xlabel('Date')
plt.ylabel('Bitcoin Price (USD)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Normalize the data for machine learning models
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_close = scaler.fit_transform(model_data[['Close']])